# Leukemia Detection — EfficientNetB0 Method 2: Fine-Tuning

**Approach:** Phase 1: Frozen EfficientNetB0 base, train classifier.
Phase 2: Unfreeze Last 20 layers, train with lower learning rate.
No data augmentation. Pickle checkpointing for Kaggle session recovery.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, pickle
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Data Loading

In [ ]:
# Dataset paths
train_dir = '/kaggle/input/datasets/avk256/cnmc-leukemia/fold_0'
val_dir = '/kaggle/input/datasets/avk256/cnmc-leukemia/fold_1'
test_dir = '/kaggle/input/datasets/avk256/cnmc-leukemia/fold_2'

for name, path in [('Train', train_dir), ('Validation', val_dir), ('Test', test_dir)]:
    if os.path.exists(path):
        print(f'\n{name} set:')
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                print(f'  {cls}: {len(os.listdir(cls_path))} images')

## Data Generators (No Augmentation)

In [ ]:
# Data generators — rescaling ONLY, no augmentation
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = datagen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True)
val_gen = datagen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)
test_gen = datagen.flow_from_directory(test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)

print(f'Class mapping: {train_gen.class_indices}')
print(f'Training: {train_gen.samples} | Validation: {val_gen.samples} | Test: {test_gen.samples}')

## Build EfficientNetB0 Model (Frozen Base)

In [ ]:
# Build model with frozen base
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Phase 1: Feature Extraction (Frozen Base)

In [ ]:
# Training with pickle checkpointing
EPOCHS = 15
CKPT = '/kaggle/working/efficientnetb0_m2_p1_ckpt.pkl'
WGTS = '/kaggle/working/efficientnetb0_m2_p1_weights.h5'

hist = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
start = 0

if os.path.exists(CKPT):
    with open(CKPT, 'rb') as f:
        ckpt = pickle.load(f)
    model.load_weights(WGTS)
    start = ckpt['epoch']
    hist = ckpt['history']
    print(f'Resuming from epoch {start}')

for epoch in range(start, EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    h = model.fit(train_gen, epochs=epoch+1, initial_epoch=epoch, validation_data=val_gen)
    for k in hist:
        hist[k].extend(h.history[k])
    model.save_weights(WGTS)
    with open(CKPT, 'wb') as f:
        pickle.dump({'epoch': epoch+1, 'history': hist}, f)

print(f'\nBest Val Accuracy: {max(hist["val_accuracy"])*100:.2f}%')

## Phase 2: Fine-Tuning (Last 20 layers)

In [ ]:
# Phase 2: Fine-Tuning — unfreeze top layers
base_model.trainable = True
ft_at = len(base_model.layers) - 20
for layer in base_model.layers[:ft_at]:
    layer.trainable = False
print(f'Trainable: {sum(1 for l in base_model.layers if l.trainable)} | Frozen: {sum(1 for l in base_model.layers if not l.trainable)}')

model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

P2_EPOCHS = 15
P2_CKPT = '/kaggle/working/efficientnetb0_m2_p2_ckpt.pkl'
P2_WGTS = '/kaggle/working/efficientnetb0_m2_p2_weights.h5'

p2_hist = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
p2_start = 0

if os.path.exists(P2_CKPT):
    with open(P2_CKPT, 'rb') as f:
        ckpt = pickle.load(f)
    model.load_weights(P2_WGTS)
    p2_start = ckpt['epoch']
    p2_hist = ckpt['history']
    print(f'Resuming Phase 2 from epoch {p2_start}')

print('PHASE 2: Fine-Tuning')
print('=' * 50)

for epoch in range(p2_start, P2_EPOCHS):
    print(f'\nEpoch {epoch+1}/{P2_EPOCHS}')
    h = model.fit(train_gen, epochs=epoch+1, initial_epoch=epoch, validation_data=val_gen)
    for k in p2_hist:
        p2_hist[k].extend(h.history[k])
    model.save_weights(P2_WGTS)
    with open(P2_CKPT, 'wb') as f:
        pickle.dump({'epoch': epoch+1, 'history': p2_hist}, f)

print(f'\nBest Phase 2 Val Accuracy: {max(p2_hist["val_accuracy"])*100:.2f}%')

## Training Curves (Both Phases)

In [ ]:
# Plot training curves — both phases
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(hist['accuracy'], label='Train'); axes[0,0].plot(hist['val_accuracy'], label='Val')
axes[0,0].set_title('Phase 1 — Accuracy', fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(hist['loss'], label='Train'); axes[0,1].plot(hist['val_loss'], label='Val')
axes[0,1].set_title('Phase 1 — Loss', fontweight='bold'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(p2_hist['accuracy'], label='Train', color='#e74c3c'); axes[1,0].plot(p2_hist['val_accuracy'], label='Val', color='#3498db')
axes[1,0].set_title('Phase 2 — Accuracy', fontweight='bold'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(p2_hist['loss'], label='Train', color='#e74c3c'); axes[1,1].plot(p2_hist['val_loss'], label='Val', color='#3498db')
axes[1,1].set_title('Phase 2 — Loss', fontweight='bold'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.suptitle('EfficientNetB0 Fine-Tuning', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

## Evaluation

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_gen)
print(f'\nTest Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

# Confusion Matrix
test_gen.reset()
y_pred = (model.predict(test_gen) > 0.5).astype(int).flatten()
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 16})
plt.title('Confusion Matrix — EfficientNetB0 (Fine-Tuned)', fontsize=16, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

## Prediction Visualization

In [ ]:
# Visualize predictions
test_gen.reset()
images, labels = next(test_gen)
predictions = model.predict(images)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    true_label = class_names[int(labels[i])]
    pred_label = class_names[int(predictions[i] > 0.5)]
    confidence = predictions[i][0] if predictions[i] > 0.5 else 1 - predictions[i][0]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1%})', fontsize=10, color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('EfficientNetB0 — Fine-Tuned Predictions', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

## Save Model

In [ ]:
model.save('efficientnetb0_method2_finetuned.keras')
print('Model saved as efficientnetb0_method2_finetuned.keras')